# GRU & Transformer 학습 결과 분석

## 목적
- Optuna TPE 최적화 결과 분석
- 버전별 (v5.5, v5.6, v5.7) 학습 성능 비교
- GRU vs Transformer 모델 비교
- 최적 하이퍼파라미터 도출

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import torch
from pathlib import Path

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
sns.set_palette('husl')

BASE_DIR = Path('c:/Users/Public/Documents/DIK/deTACTer')
MODELS_DIR = BASE_DIR / 'results' / 'models'
VERSIONS = ['v5.5', 'v5.6', 'v5.7']

## 1. Optuna Study 로드 및 기본 통계

In [ ]:
# GRU Studies 로드
gru_studies = {}
for version in VERSIONS:
    db_path = MODELS_DIR / version / f'gru_{version}.db'
    if db_path.exists():
        storage = f'sqlite:///{db_path}'
        study = optuna.load_study(study_name=f'gru_{version}', storage=storage)
        gru_studies[version] = study
        print(f"\n{version} GRU Study:")
        print(f"  Total trials: {len(study.trials)}")
        print(f"  Best value: {study.best_value:.6f}")
        print(f"  Best params: {study.best_params}")

In [ ]:
# Transformer Studies 로드
transformer_studies = {}
for version in VERSIONS:
    db_path = MODELS_DIR / version / f'transformer_{version}.db'
    if db_path.exists():
        storage = f'sqlite:///{db_path}'
        try:
            study = optuna.load_study(study_name=f'transformer_{version}', storage=storage)
            transformer_studies[version] = study
            print(f"\n{version} Transformer Study:")
            print(f"  Total trials: {len(study.trials)}")
            print(f"  Best value: {study.best_value:.6f}")
            print(f"  Best params: {study.best_params}")
        except:
            print(f"\n{version} Transformer: 학습 미완료 또는 DB 없음")

## 2. 학습 수렴 곡선 (Optimization History)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, version in enumerate(VERSIONS):
    ax = axes[idx]
    
    # GRU
    if version in gru_studies:
        study = gru_studies[version]
        trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
        values = [t.value for t in trials]
        ax.plot(values, 'o-', label='GRU', alpha=0.6, markersize=3)
        
        # Best value line
        best_values = np.minimum.accumulate(values)
        ax.plot(best_values, 'r-', linewidth=2, label='GRU Best', alpha=0.8)
    
    # Transformer
    if version in transformer_studies:
        study = transformer_studies[version]
        trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
        values = [t.value for t in trials]
        ax.plot(values, 's-', label='Transformer', alpha=0.6, markersize=3)
        
        best_values = np.minimum.accumulate(values)
        ax.plot(best_values, 'g-', linewidth=2, label='Transformer Best', alpha=0.8)
    
    ax.set_xlabel('Trial Number')
    ax.set_ylabel('Validation Loss')
    ax.set_title(f'{version} Optimization History')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(BASE_DIR / 'results' / 'models' / 'optimization_history.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. 버전별 최종 성능 비교

In [ ]:
# 성능 비교 테이블
results = []

for version in VERSIONS:
    row = {'Version': version}
    
    if version in gru_studies:
        row['GRU Best Loss'] = gru_studies[version].best_value
        row['GRU Trials'] = len([t for t in gru_studies[version].trials 
                                 if t.state == optuna.trial.TrialState.COMPLETE])
    
    if version in transformer_studies:
        row['Transformer Best Loss'] = transformer_studies[version].best_value
        row['Transformer Trials'] = len([t for t in transformer_studies[version].trials 
                                         if t.state == optuna.trial.TrialState.COMPLETE])
    
    results.append(row)

results_df = pd.DataFrame(results)
print("\n=" * 80)
print("버전별 최종 성능 비교")
print("=" * 80)
print(results_df.to_string(index=False))
print("=" * 80)

In [ ]:
# 시각화
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(VERSIONS))
width = 0.35

gru_losses = [results_df.loc[results_df['Version'] == v, 'GRU Best Loss'].values[0] 
              if 'GRU Best Loss' in results_df.columns and not pd.isna(results_df.loc[results_df['Version'] == v, 'GRU Best Loss'].values[0])
              else 0 for v in VERSIONS]

transformer_losses = [results_df.loc[results_df['Version'] == v, 'Transformer Best Loss'].values[0] 
                      if 'Transformer Best Loss' in results_df.columns and not pd.isna(results_df.loc[results_df['Version'] == v, 'Transformer Best Loss'].values[0])
                      else 0 for v in VERSIONS]

ax.bar(x - width/2, gru_losses, width, label='GRU', alpha=0.8)
ax.bar(x + width/2, transformer_losses, width, label='Transformer', alpha=0.8)

ax.set_xlabel('Version')
ax.set_ylabel('Best Validation Loss')
ax.set_title('버전별 모델 성능 비교')
ax.set_xticks(x)
ax.set_xticklabels(VERSIONS)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(BASE_DIR / 'results' / 'models' / 'version_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. 최적 하이퍼파라미터 분석

In [ ]:
# GRU 최적 파라미터 비교
gru_params_df = pd.DataFrame([
    {'Version': version, **gru_studies[version].best_params}
    for version in VERSIONS if version in gru_studies
])

print("\nGRU 최적 하이퍼파라미터:")
print(gru_params_df.to_string(index=False))

In [ ]:
# Transformer 최적 파라미터 비교 (학습 완료된 버전만)
if transformer_studies:
    transformer_params_df = pd.DataFrame([
        {'Version': version, **transformer_studies[version].best_params}
        for version in VERSIONS if version in transformer_studies
    ])
    
    print("\nTransformer 최적 하이퍼파라미터:")
    print(transformer_params_df.to_string(index=False))

## 5. 모델 파일 정보

In [ ]:
# 저장된 모델 파일 확인
model_files = []

for version in VERSIONS:
    for model_type in ['gru', 'transformer']:
        model_path = MODELS_DIR / version / f'{model_type}_best.pth'
        if model_path.exists():
            checkpoint = torch.load(model_path, map_location='cpu')
            model_files.append({
                'Version': version,
                'Model': model_type.upper(),
                'Best Val Loss': checkpoint['best_val_loss'],
                'Max Seq Len': checkpoint['max_len'],
                'File Size (MB)': model_path.stat().st_size / (1024*1024)
            })

model_files_df = pd.DataFrame(model_files)
print("\n저장된 모델 파일:")
print(model_files_df.to_string(index=False))

## 6. 결론 및 인사이트

### 주요 발견사항
1. **버전별 성능**: (위 결과 참조)
2. **GRU vs Transformer**: (위 결과 참조)
3. **최적 하이퍼파라미터 패턴**: (위 결과 참조)

### 다음 단계
- 임베딩 기반 클러스터링 수행
- OPTICS 좌표 기반 클러스터링과 비교
- 전술 패턴 분석